# 📉 InvestAI — Módulo 4: Regresor LSTM (Predicción de Precios)
**Entrada:** MongoDB `precios_ohlcv` · **Salida:** `predicciones_lstm`, `metricas_modelos`
**Arquitectura:** `LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16) → Dense(1)`
**Ventana:** 60 días (fija) · **Horizontes:** 7, 14, 30, 60 días · **Confianza:** 95% (±1.96×RMSE)
> ⚠️ Activar GPU: Entorno → T4 GPU. Requiere Secret `MONGO_URI`. Ejecutar después del Módulo 1.

In [ ]:
import requests

# Obtener la IP pública actual de la sesión de Colab
response = requests.get('https://api.ipify.org?format=json')
public_ip = response.json()['ip']
print(f"Tu dirección IP pública actual de Colab es: {public_ip}")

Tu dirección IP pública actual de Colab es: 34.81.114.169


## Sección 1 — Instalación e Importaciones

In [ ]:
!pip install "pymongo[srv]" --quiet
import tensorflow as tf
print(f"✅ TensorFlow {tf.__version__}")
print(f"   GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
import warnings, math
warnings.filterwarnings('ignore')
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
from google.colab import userdata

tf.random.set_seed(42)
np.random.seed(42)
print("✅ Importaciones OK.")

## Sección 2 — Configuración

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
EMPRESAS_META = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',             'moneda': 'USD'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compania Minera S.A.A.',         'moneda': 'PEN'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',              'moneda': 'CAD'},
    'BVN':         {'nombre': 'Compania de Minas Buenaventura S.A.A.', 'moneda': 'USD'},
    'BHP':         {'nombre': 'BHP Group Limited',                     'moneda': 'USD'},
}

VENTANA       = 60          # Días de historia por muestra (NO modificar)
HORIZONTES    = [7, 14, 30, 60]
TEST_SPLIT    = 0.15
EPOCHS        = 100
BATCH_SIZE    = 32
LEARNING_RATE = 0.001
PATIENCE_ES   = 15
PATIENCE_LR   = 7
MIN_REGISTROS = VENTANA * 3  # ~180 registros mínimos

DB_NOMBRE        = 'investai'
COL_PRECIOS      = 'precios_ohlcv'
COL_PRED_LSTM    = 'predicciones_lstm'
COL_METRICAS     = 'metricas_modelos'

print(f"Ventana: {VENTANA} días (fija)  |  Horizontes: {HORIZONTES} días")
print(f"Test split: {TEST_SPLIT*100:.0f}%  |  Épocas máx: {EPOCHS}")
print("✅ Configuración OK.")

## Sección 3 — Conexión a MongoDB

In [ ]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("Secret MONGO_URI vacío.")
except Exception as e:
    raise RuntimeError(f"No se encontró MONGO_URI: {e}")

try:
    cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
    cliente.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida.")
except ConnectionFailure as e:
    raise RuntimeError(f"Falla de conexión: {e}")

db            = cliente[DB_NOMBRE]
col_precios   = db[COL_PRECIOS]
col_pred_lstm = db[COL_PRED_LSTM]
col_metricas  = db[COL_METRICAS]

col_pred_lstm.create_index([('ticker', 1)], unique=True, name='idx_ticker')

n = col_precios.count_documents({})
print(f"   precios_ohlcv: {n:,} documentos")
if n == 0:
    print("⚠️  Ejecuta primero el Módulo 1 (Ingesta de Datos).")

## Sección 4 — Funciones de Utilidad

In [ ]:
def nan_a_none(v):
    if v is None: return None
    try:
        return None if (math.isnan(float(v)) or math.isinf(float(v))) else round(float(v), 6)
    except (TypeError, ValueError):
        return None


def leer_serie(ticker):
    """Lee la serie de cierres del ticker desde MongoDB."""
    docs = list(
        col_precios.find(
            {'ticker': ticker, 'cierre': {'$ne': None}},
            {'_id': 0, 'fecha': 1, 'cierre': 1}
        ).sort('fecha', 1)
    )
    if not docs:
        return None
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.set_index('fecha').sort_index()
    df = df[~df.index.duplicated(keep='last')]
    return df['cierre'].dropna()


def crear_ventanas(serie_norm, ventana):
    """Crea ventanas deslizantes para el modelo LSTM."""
    X, y = [], []
    for i in range(ventana, len(serie_norm)):
        X.append(serie_norm[i - ventana:i, 0])
        y.append(serie_norm[i, 0])
    return np.array(X).reshape(-1, ventana, 1), np.array(y)


def construir_modelo(ventana, lr=0.001):
    """
    Arquitectura fija del LSTM Regressor:
    LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, relu) -> Dense(1, linear)
    """
    modelo = Sequential(name='LSTM_Regressor', layers=[
        Input(shape=(ventana, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1,  activation='linear'),
    ])
    modelo.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return modelo

print("✅ Funciones de utilidad definidas.")

## Sección 5 — Pipeline de Entrenamiento por Ticker

In [ ]:
def predecir_horizonte(modelo, ult_ventana_norm, scaler, horizonte, rmse_usd):
    """
    Predicción multi-paso iterativa (rolling forecast):
    Predice día t+1, lo agrega a la ventana, repite 'horizonte' veces.
    """
    ventana_actual = ult_ventana_norm.copy().reshape(-1, 1)
    preds_norm = []
    for _ in range(horizonte):
        entrada = ventana_actual[-VENTANA:].reshape(1, VENTANA, 1)
        pred_n  = modelo.predict(entrada, verbose=0)[0, 0]
        preds_norm.append(pred_n)
        ventana_actual = np.append(ventana_actual, [[pred_n]], axis=0)

    preds_usd = scaler.inverse_transform(
        np.array(preds_norm).reshape(-1, 1)
    ).flatten().tolist()

    return {
        'predicciones'  : [nan_a_none(p) for p in preds_usd],
        'banda_superior': [nan_a_none(p + 1.96 * rmse_usd) for p in preds_usd],
        'banda_inferior': [nan_a_none(p - 1.96 * rmse_usd) for p in preds_usd],
        'precio_final'  : nan_a_none(preds_usd[-1]),
    }


def entrenar_lstm(ticker):
    # 1. Leer serie
    serie = leer_serie(ticker)
    if serie is None or len(serie) < MIN_REGISTROS:
        n = 0 if serie is None else len(serie)
        print(f"   ⚠️  {ticker}: {n} registros — se requieren >= {MIN_REGISTROS}.")
        return None

    valores = serie.values.reshape(-1, 1).astype(float)

    # 2. Partición temporal ANTES de normalizar (evita leakage del scaler)
    corte     = int(len(valores) * (1 - TEST_SPLIT))
    train_raw = valores[:corte]
    test_raw  = valores[corte:]

    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw)   # fit SOLO en train
    test_norm  = scaler.transform(test_raw)        # transform en test

    todo_norm = np.concatenate([train_norm, test_norm], axis=0)

    # 3. Ventanas deslizantes
    X, y = crear_ventanas(todo_norm, VENTANA)

    # 4. Split train/test de ventanas
    n_train = corte - VENTANA
    if n_train <= 0:
        print(f"   ⚠️  {ticker}: datos insuficientes para ventana de {VENTANA}.")
        return None

    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]

    if len(X_test) < 5:
        print(f"   ⚠️  {ticker}: test insuficiente ({len(X_test)} muestras).")
        return None

    print(f"   Train: {X_train.shape}  Test: {X_test.shape}")

    # 5. Construir y entrenar
    modelo = construir_modelo(VENTANA, LEARNING_RATE)
    cbs = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', patience=PATIENCE_LR,
                          factor=0.5, min_lr=1e-6, verbose=0),
    ]
    hist = modelo.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1, callbacks=cbs, verbose=1
    )
    epocas_reales = len(hist.history['loss'])

    # 6. Métricas en test
    y_pred_n   = modelo.predict(X_test, verbose=0).flatten()
    y_real_usd = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_usd = scaler.inverse_transform(y_pred_n.reshape(-1, 1)).flatten()

    rmse_usd  = float(np.sqrt(mean_squared_error(y_real_usd, y_pred_usd)))
    precio_med = float(np.mean(y_real_usd))
    rmse_pct  = (rmse_usd / precio_med * 100) if precio_med != 0 else 0.0
    mae_usd   = float(mean_absolute_error(y_real_usd, y_pred_usd))
    r2        = float(r2_score(y_real_usd, y_pred_usd))

    print(f"   RMSE: ${rmse_usd:.4f} ({rmse_pct:.2f}%)  MAE: ${mae_usd:.4f}  R2: {r2:.4f}")

    # 7. Predicciones futuras por horizonte
    ult_ventana = todo_norm[-VENTANA:].flatten()
    fecha_base  = serie.index[-1]
    preds_fut   = {}
    for h in HORIZONTES:
        ph = predecir_horizonte(modelo, ult_ventana, scaler, h, rmse_usd)
        fechas_fut = [
            (fecha_base + timedelta(days=i+1)).strftime('%Y-%m-%d')
            for i in range(h)
        ]
        preds_fut[str(h)] = {
            'fechas': fechas_fut,
            'predicciones': ph['predicciones'],
            'banda_superior': ph['banda_superior'],
            'banda_inferior': ph['banda_inferior'],
            'precio_final': ph['precio_final'],
        }
        print(f"   +{h:>2}d → {ph['precio_final']}")

    # 8. Historial test para visualización
    fechas_test = serie.index[corte:].strftime('%Y-%m-%d').tolist()
    min_len = min(len(fechas_test), len(y_real_usd))

    return {
        'ticker': ticker,
        'ultimo_precio': nan_a_none(float(serie.iloc[-1])),
        'fecha_ultimo': serie.index[-1].strftime('%Y-%m-%d'),
        'n_train': len(X_train), 'n_test': len(X_test),
        'epocas_reales': epocas_reales,
        'metricas': {
            'rmse_usd': nan_a_none(rmse_usd), 'rmse_pct': nan_a_none(rmse_pct),
            'mae_usd':  nan_a_none(mae_usd),  'r2': nan_a_none(r2),
        },
        'predicciones_futuras': preds_fut,
        'historico_test': {
            'fechas':    fechas_test[-min_len:],
            'real_usd':  [nan_a_none(v) for v in y_real_usd[-min_len:].tolist()],
            'pred_usd':  [nan_a_none(v) for v in y_pred_usd[-min_len:].tolist()],
            'banda_sup': [nan_a_none(v + 1.96*rmse_usd) for v in y_pred_usd[-min_len:].tolist()],
            'banda_inf': [nan_a_none(v - 1.96*rmse_usd) for v in y_pred_usd[-min_len:].tolist()],
        },
    }

print("✅ Función entrenar_lstm() definida.")

## Sección 6 — Persistencia y Pipeline Principal

In [ ]:
def guardar_lstm(res):
    meta = EMPRESAS_META.get(res['ticker'], {})
    doc = {
        'ticker': res['ticker'], 'nombre': meta.get('nombre', res['ticker']),
        'moneda': meta.get('moneda', 'USD'), 'modelo': 'LSTM_Regressor',
        'arquitectura': 'LSTM(64)->Dropout->LSTM(32)->Dropout->Dense(16)->Dense(1)',
        'ventana_dias': VENTANA, 'horizontes': HORIZONTES,
        'ultimo_precio': res['ultimo_precio'], 'fecha_ultimo': res['fecha_ultimo'],
        'metricas': res['metricas'],
        'predicciones_futuras': res['predicciones_futuras'],
        'historico_test': res['historico_test'],
        'n_train': res['n_train'], 'n_test': res['n_test'],
        'epocas_reales': res['epocas_reales'],
        'nivel_confianza': '95% (+/-1.96xRMSE)',
        'actualizado_en': datetime.now(timezone.utc),
    }
    col_pred_lstm.update_one({'ticker': res['ticker']}, {'$set': doc}, upsert=True)

def guardar_met_lstm(res):
    meta = EMPRESAS_META.get(res['ticker'], {})
    col_metricas.insert_one({
        'ticker': res['ticker'], 'nombre': meta.get('nombre', res['ticker']),
        'modelo': 'LSTM_Regressor', 'tipo_tarea': 'regresion',
        'ventana_dias': VENTANA, 'horizontes': HORIZONTES,
        'n_train': res['n_train'], 'n_test': res['n_test'],
        'epocas_reales': res['epocas_reales'],
        'metricas': res['metricas'],
        'entrenado_en': datetime.now(timezone.utc),
    })

print("✅ Funciones de persistencia definidas.")

In [ ]:
resumen = []
print("=" * 60)
print("  REGRESOR LSTM — InvestAI")
print("=" * 60)

for ticker in TICKERS:
    meta = EMPRESAS_META.get(ticker, {})
    print(f"\n{'='*55}")
    print(f"  📉 {ticker}  ({meta.get('nombre','')})")
    print(f"{'='*55}")

    res = entrenar_lstm(ticker)
    if res is None:
        resumen.append({'ticker': ticker, 'estado': '❌'})
        continue

    guardar_lstm(res)
    guardar_met_lstm(res)
    print("   ✅ Guardado en MongoDB.")

    resumen.append({
        'ticker': ticker, 'estado': '✅',
        'rmse_usd': res['metricas']['rmse_usd'],
        'rmse_pct': res['metricas']['rmse_pct'],
        'r2': res['metricas']['r2'],
        '+7d': res['predicciones_futuras']['7']['precio_final'],
        '+30d': res['predicciones_futuras']['30']['precio_final'],
    })

print("\n" + "=" * 60)
print(pd.DataFrame(resumen).to_string(index=False))
print("=" * 60)

## Sección 7 — Verificación Final

In [ ]:
n_pred = col_pred_lstm.count_documents({})
n_met  = col_metricas.count_documents({'modelo': 'LSTM_Regressor'})
print(f"predicciones_lstm : {n_pred} docs (esperado: {len(TICKERS)})")
print(f"metricas_modelos  : {n_met} docs LSTM")

print("\nPredicciones por ticker:")
for doc in col_pred_lstm.find({}, {'_id':0,'historico_test':0}).sort('ticker',1):
    m  = doc.get('metricas', {})
    p7 = doc.get('predicciones_futuras',{}).get('7',{}).get('precio_final','?')
    print(f"  {doc['ticker']:<14}  ultimo: {doc.get('ultimo_precio')}  "
          f"RMSE: {m.get('rmse_pct','?')}%  R2: {m.get('r2','?')}  +7d: {p7}")

cliente.close()
print("\n✅ Verificación OK. Conexión cerrada.")